In [2]:
# import packages
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris

In [26]:
# import cleaned fire dataset

project_fldr = Path.cwd().parents[1]

raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# read in CSV
fire_shp_25 = gpd.read_file(processed_fldr / "fire_shp_25/fire_shp_25.shp")
fire_shp_25.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   YEAR_       521 non-null    float64 
 1   STATE       521 non-null    str     
 2   AGENCY      521 non-null    str     
 3   UNIT_ID     521 non-null    str     
 4   FIRE_NAME   521 non-null    str     
 5   INC_NUM     521 non-null    str     
 6   IRWINID     469 non-null    str     
 7   ALARM_DATE  521 non-null    str     
 8   CONT_DATE   514 non-null    str     
 9   C_METHOD    521 non-null    int32   
 10  CAUSE       521 non-null    int32   
 11  OBJECTIVE   521 non-null    float64 
 12  GIS_ACRES   521 non-null    float64 
 13  COMPLEX_ID  17 non-null     str     
 14  COMPLEX_NA  17 non-null     str     
 15  COMMENTS    120 non-null    str     
 16  FIRE_NUM    0 non-null      object  
 17  Shape_Leng  521 non-null    float64 
 18  Shape_Area  521 non-null    float64 
 19  

In [42]:
fire_shp_25['CAUSE'].value_counts()

CAUSE
14    202
1      98
2      63
10     41
7      30
11     26
9      26
5      22
4      10
8       1
15      1
6       1
Name: count, dtype: int64

In [36]:
# make alarm date column datetime
fire_shp_25['ALARM_DATE'] = pd.to_datetime(fire_shp_25['ALARM_DATE'])
fire_shp_25['CONT_DATE'] = pd.to_datetime(fire_shp_25['CONT_DATE'])

In [43]:
#map cause to code
cause_to_code = {
    "Lightning": 1,
    "Equipment Use": 2,
    "Smoking": 3,
    "Campfire": 4,
    "Debris Burning": 5,
    "Railroad": 6,
    "Arson": 7,
    "Children (Fire Play)": 8,
    "Miscellaneous / Other": 9,
    "Fireworks": 10,
    "Power Line": 11,
    "Structure": 12,
    "Ammunition / Explosives": 13,
    "Unknown (Human / Undetermined)": 14,
}

In [ ]:
# Set the "selected" values
alarm_start_date = "2025-08-01" #alarm date dropdown range
alarm_end_date = "2025-12-31"
cont_start_date = "2025-08-01" #containment date dropdown range
cont_end_date = "2025-12-31"
cause = 14 # there are 14 different causes, we will make the dropdown by actual cause instead of codes
size = 0

# Subset the data
fire_subset = fire_shp_25.loc[
    (fire_shp_25['ALARM_DATE'] >= alarm_start_date) &  # Start date
    (fire_shp_25['ALARM_DATE'] <= alarm_end_date) &    # End date
    (fire_shp_25['CONT_DATE'] >= cont_start_date) &    # Containment start date
    (fire_shp_25['CONT_DATE'] <= cont_end_date) &      # Containment end date
    (fire_shp_25['CAUSE'] == cause) &  
    (fire_shp_25['Shape_Area'] >= size),  
]

fire_subset.head()


,YEAR_,STATE,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,IRWINID,ALARM_DATE,CONT_DATE,C_METHOD,...,COMPLEX_ID,COMPLEX_NA,COMMENTS,FIRE_NUM,Shape_Leng,Shape_Area,GlobalID,Alarm Mont,Alarm Mo_1,geometry
0,2025.0,CA,USF,LPF,GIFFORD,00002181,{08A6358F-8301-4093-9988-E265C8F6F0A9},2025-08-01 00:00:00+00:00,2025-10-14 00:00:00+00:00,3,...,NaN,NaN,NaN,None,257388.320379,5.326211e+08,{E87207FF-BD3D-4836-8A89-AD218A20B17E},8.0,Aug,"MULTIPOLYGON (((-35131.146 -302750.005, -35107..."
11,2025.0,CA,CDF,LNU,PICKETT,00015521,{D7CD7F3F-C6E1-48EF-BABE-E88A868A4F68},2025-08-21 00:00:00+00:00,2025-09-07 00:00:00+00:00,1,...,NaN,NaN,NaN,None,57212.665363,2.760883e+07,{C465575B-0B61-4908-A241-8180B23FAC39},8.0,Aug,"MULTIPOLYGON (((-215157.954 73534.501, -215156..."
13,2025.0,CA,CCO,VNC,CANYON,00068268,{66BE6702-3F5F-463C-B0E8-6D8BEA38EEC4},2025-08-06 00:00:00+00:00,2025-08-11 00:00:00+00:00,2,...,NaN,NaN,NaN,None,46638.405298,2.172240e+07,{A71D2959-C175-42D2-A4A7-6C6A125F16A1},8.0,Aug,"MULTIPOLYGON (((117029.871 -392760.514, 117043..."
21,2025.0,CA,USF,SQF,LITTLE,00001327,{6B49F156-AA48-475D-91C3-915222C8B2A2},2025-08-22 00:00:00+00:00,2025-09-01 00:00:00+00:00,7,...,NaN,NaN,"CAKRN-038265, origin destroyed",None,23872.398784,1.014364e+07,{338DE310-1602-479F-B998-7ABADB709840},8.0,Aug,"MULTIPOLYGON (((126417.504 -267390.788, 126441..."
25,2025.0,CA,CDF,RRU,ROSA,00122253,{37317992-17AD-469F-8E36-5BC2D678740F},2025-08-04 00:00:00+00:00,2025-08-12 00:00:00+00:00,7,...,NaN,NaN,NaN,None,21657.954526,6.761883e+06,{62D979B3-9EB9-453E-8FFC-4D2A68DA7A83},8.0,Aug,"MULTIPOLYGON (((325590.284 -485852.665, 325590..."
